# EEG sleep foundation — train on a Colab GPU

Track 03 (Sleep Onset). This notebook rebuilds the project on Colab and trains
the submissions there:

1. **Runtime → Change runtime type → GPU** (T4 is enough), then **Runtime → Run all**.
2. The Muse data (~1.1 GB) is downloaded from NEMAR on the first run and cached
   in Google Drive, so later sessions only extract it.
3. Each finished build (`build/<label>/`, `dist/<label>.zip`, the log) is copied
   to Drive right away, so a disconnect does not lose finished runs.

Generated by `scripts/make_colab_notebook.py`; regenerate it after editing the
solver locally.

In [ ]:
!nvidia-smi -L
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Select a GPU runtime first."

## Settings

In [ ]:
from pathlib import Path

PROJECT = Path("/content/EEG_sleep_foundation")
COMPETITION_REPO = "https://github.com/neural-interfaces26/2026-competition.git"
COMPETITION_COMMIT = "a66bc23d1a0e686b895c093a642b45c74970758b"   # the harness version used locally

USE_DRIVE = True    # cache the data and save the results in Google Drive
DRIVE_DIR = Path("/content/drive/MyDrive/EEG_sleep_foundation")

In [ ]:
# Drive can only be mounted by the Google account running this Colab session.
# Picking a different account in the popup fails with "credential
# propagation was unsuccessful": open this notebook while signed in to Colab
# as the account whose Drive you want to use.
if USE_DRIVE:
    from google.colab import drive
    try:
        drive.mount("/content/drive")
        DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    except Exception as err:
        USE_DRIVE = False
        print(f"Drive not mounted ({err}).\n"
              "Continuing WITHOUT Drive: the data is downloaded again and "
              "results are not saved if the runtime disconnects. Download "
              "them with the last cell.\n"
              "To use Drive, run this notebook in Colab signed in as the "
              "account that owns that Drive.")

## Code: official harness + this project's files

In [ ]:
import subprocess
if not (PROJECT / "competition").exists():
    subprocess.run(["git", "clone", "-q", COMPETITION_REPO,
                    str(PROJECT / "competition")], check=True)
subprocess.run(["git", "-C", str(PROJECT / "competition"), "checkout", "-q",
                COMPETITION_COMMIT], check=True)
for sub in ("submission", "scripts", "build", "dist"):
    (PROJECT / sub).mkdir(parents=True, exist_ok=True)
%cd {PROJECT}

In [ ]:
%%writefile /content/EEG_sleep_foundation/submission/cnn_gru.py
"""Track 03 (Sleep Onset) submission: an ensemble of spectral-feature -> GRU
models that each output a time-to-N2 distribution.

Each 5 s window is summarised by hand-crafted spectral features:
- log band powers (delta, theta, alpha, sigma, beta, high/EMG), absolute and
  relative, and the theta/alpha ratio: the classic sleep-scoring markers;
- with ``feats=2`` (default) also drowsiness markers: slow (< 0.6 Hz) power
  per channel and on the AF7-AF8 derivation (slow eye movements), 0.6-4 Hz
  power on AF7-AF8, spectral entropy and 95 % spectral edge per channel.
Optionally (``use_cnn``, off by default since the bottleneck experiments
showed it does not help with this much data) a 1-D CNN on the raw signal
adds learned waveform features.
These, plus the time elapsed since the recording started, feed a GRU that
keeps its state across the windows of a recording (streamed evaluation: one
window per ``predict`` call, ``reset_state`` per recording).

Per-sleeper adaptation (``adapt``): the features are also given relative to
the recording's running mean, so each window is judged against the same
sleeper's history; with ``feats=2`` also relative to a ~1 min EMA (the
short-term change) and the EMA relative to the running mean (the trend).
All statistics are causal, over the windows seen so far.

The head outputs a distribution over the time to the first N2 epoch, in
5 s bins (0, 5, ..., 595, 600 = capped). The ensemble averages the members'
distributions; ``p[0]`` is the hazard (probability that N2 starts within the
next window) and the prediction is the ``q``-quantile of the distribution,
with ``q`` tuned on held-out subjects.

Contract (see competition/tracks/sleep_onset/README.md):
- input  X: float tensor (B, C, T) on meta["device"], raw microvolts
- output  : (B,) float latencies in seconds to the first N2 epoch, in [0, 600]

This file must stay self-contained: only ``torch`` and packages from the
competition's requirements.txt may be imported, and no other local modules.
"""

import os
import time

import numpy as np
import torch
from torch import nn

from benchmark_utils.base_solver import CompetSolver

CAP_S = 600.0
WEIGHTS_FILE = "weights.pt"

BIN_S = 5.0
N_BINS = int(CAP_S / BIN_S) + 1          # 0, 5, ..., 600 s
CLIP_UV = 500.0
BANDS = ((0.5, 4.0), (4.0, 8.0), (8.0, 12.0), (12.0, 15.0), (15.0, 30.0),
         (30.0, 60.0))
# Metric ranges and severity weights (objective.py).
RANGE_EDGES = (0.0, 40.0, 90.0, 300.0, 600.0)
RANGE_WEIGHTS = (10.0, 5.0, 3.0, 1.0)

CNN_CHANNELS = (32, 64, 96, 128)
BP_DIM = 64
GRU_HIDDEN = 128
GRU_LAYERS = 2
DROPOUT = 0.2

# Training-time choices; inference reads them back from weights.pt.
N_SEEDS = int(os.environ.get("SLEEP_N_SEEDS", 5))
ADAPT = os.environ.get("SLEEP_ADAPT", "1") == "1"
# Raw-signal CNN branch (off: it did not help in the bottleneck experiments)
# and the feature set (1 = band powers only, 2 = + drowsiness and trend).
USE_CNN = os.environ.get("SLEEP_USE_CNN", "0") == "1"
FEATS = int(os.environ.get("SLEEP_FEATS", 2))
# Also train on NeuralBench's own validation sleepers of the public Muse
# training release, which the harness's train loader leaves out.
EXTRA_DATA = os.environ.get("SLEEP_EXTRA_DATA", "1") == "1"
# Diagnostics (see colab notebook): train on a fraction of the training
# sleepers, change the members' seed. The validation sleepers do not depend
# on any of these settings, so runs stay comparable.
TRAIN_FRAC = float(os.environ.get("SLEEP_TRAIN_FRAC", 1.0))
MEMBER_SEED = int(os.environ.get("SLEEP_SEED", 0))
EMA_S = 60.0                             # short-term EMA time constant

EPOCHS = 30
PATIENCE = 8
LR = 1e-3
WEIGHT_DECAY = 1e-4
RECS_PER_BATCH = 16
CHUNK = 120                              # TBPTT length, in windows (10 min)
VAL_FRACTION = 0.15
LABEL_SIGMA = 1.5                        # soft-label width, in bins
L1_WEIGHT = 0.1
QUANTILES = (0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6)
SEED = 0                                 # validation split seed (fixed)


def preprocess(X):
    """Raw microvolts (..., C, T) -> per-window DC removed, artifacts clipped."""
    X = X - X.mean(dim=-1, keepdim=True)
    return X.clamp(-CLIP_UV, CLIP_UV)


def quantile_seconds(p, q):
    """Distributions (..., K) over the 5 s bins -> their q-quantile, seconds."""
    cdf = p.cumsum(-1).contiguous()
    k = torch.searchsorted(cdf, torch.full(cdf.shape[:-1] + (1,), float(q),
                                           device=cdf.device))
    return (k.clamp_max(N_BINS - 1).squeeze(-1) * BIN_S).float()


class SleepOnsetCNNGRU(nn.Module):
    """One member: (B, L, C, T) sequences -> logits (B, L, K) over bins."""

    def __init__(self, n_chans, n_times, sfreq, adapt=ADAPT, use_cnn=USE_CNN,
                 feats=FEATS):
        super().__init__()
        self.n_chans, self.n_times, self.sfreq = n_chans, n_times, sfreq
        self.window_s = n_times / sfreq
        self.use_adapt = bool(adapt)
        self.use_cnn = bool(use_cnn)
        self.n_feats = int(feats)
        self.ema_alpha = min(1.0, self.window_s / EMA_S)
        self.register_buffer("adapt", torch.tensor(self.use_adapt))
        self.register_buffer("cnn_on", torch.tensor(self.use_cnn))
        self.register_buffer("feats", torch.tensor(self.n_feats))
        self.register_buffer("chan_std", torch.ones(n_chans))
        self.register_buffer(
            "bin_values", torch.arange(N_BINS, dtype=torch.float32) * BIN_S)

        # Band masks over the rFFT bins, from the actual sampling rate.
        freqs = torch.fft.rfftfreq(n_times, d=1.0 / sfreq)
        nyq = sfreq / 2
        masks = torch.stack([((freqs >= lo) & (freqs < min(hi, nyq))).float()
                             for lo, hi in BANDS])
        self.register_buffer("band_masks", masks, persistent=False)
        self.register_buffer("hann", torch.hann_window(n_times),
                             persistent=False)
        self.register_buffer("freqs", freqs, persistent=False)
        self.register_buffer("slow_mask", ((freqs > 0) & (freqs < 0.6)).float(),
                             persistent=False)
        self.register_buffer("eog_mask", ((freqs >= 0.6) & (freqs < 4.0))
                             .float(), persistent=False)
        self.register_buffer("broad_mask", ((freqs >= 0.5)
                                            & (freqs < min(30.0, nyq))),
                             persistent=False)
        n_bp = self.band_powers(torch.zeros(1, n_chans, n_times)).shape[1]
        self.register_buffer("bp_mean", torch.zeros(n_bp))
        self.register_buffer("bp_std", torch.ones(n_bp))

        layers, in_ch = [], n_chans
        kernels = (max(3, int(sfreq // 2)) | 1, 9, 7, 5)
        for out_ch, k in zip(CNN_CHANNELS, kernels):
            layers += [nn.Conv1d(in_ch, out_ch, k, padding=k // 2, bias=False),
                       nn.BatchNorm1d(out_ch), nn.GELU(), nn.MaxPool1d(2)]
            in_ch = out_ch
        self.cnn = nn.Sequential(*layers) if self.use_cnn else None
        bp_in = n_bp * ((4 if self.n_feats >= 2 else 2) if self.use_adapt
                        else 1)
        self.bp_proj = nn.Sequential(nn.Linear(bp_in, BP_DIM), nn.GELU())
        cnn_dim = 2 * CNN_CHANNELS[-1] if self.use_cnn else 0
        self.in_proj = nn.Sequential(
            nn.Linear(cnn_dim + BP_DIM + 1, GRU_HIDDEN),
            nn.GELU(), nn.Dropout(DROPOUT))
        self.gru = nn.GRU(GRU_HIDDEN, GRU_HIDDEN, GRU_LAYERS, batch_first=True,
                          dropout=DROPOUT)
        self.head = nn.Sequential(nn.Dropout(DROPOUT),
                                  nn.Linear(GRU_HIDDEN, N_BINS))
        self.reset_state()

    def band_powers(self, X):
        """Preprocessed (N, C, T) -> raw log band-power features (N, F)."""
        spec = torch.fft.rfft(X * self.hann, dim=-1).abs() ** 2
        power = torch.einsum("nct,bt->ncb", spec, self.band_masks)
        log_abs = torch.log(power + 1e-6)
        log_rel = torch.log(power / (power.sum(-1, keepdim=True) + 1e-6)
                            + 1e-6)
        theta_alpha = log_abs[..., 1] - log_abs[..., 2]
        out = [log_abs.flatten(1), log_rel.flatten(1), theta_alpha]
        if self.n_feats >= 2:
            # Drowsiness: slow eye movements (< 0.6 Hz, strongest on the
            # AF7-AF8 derivation, where horizontal eye movements are of
            # opposite polarity), spectral flattening (entropy, edge).
            out.append(torch.log(spec @ self.slow_mask + 1e-6))
            a, b = (1, 2) if X.shape[1] >= 3 else (0, X.shape[1] - 1)
            eog = torch.fft.rfft((X[:, a] - X[:, b]) * self.hann,
                                 dim=-1).abs() ** 2
            out.append(torch.stack([torch.log(eog @ self.slow_mask + 1e-6),
                                    torch.log(eog @ self.eog_mask + 1e-6)],
                                   dim=1))
            p = spec[..., self.broad_mask]
            p = p / (p.sum(-1, keepdim=True) + 1e-12)
            out.append(-(p * torch.log(p + 1e-12)).sum(-1)
                       / np.log(p.shape[-1]))
            edge = (p.cumsum(-1) >= 0.95).float().argmax(-1)
            out.append(self.freqs[self.broad_mask][edge] / 30.0)
        return torch.cat(out, dim=1)

    def encode(self, X, start_index, sums):
        """Sequences (B, L, C, T) -> GRU inputs (B, L, H), new running sums.

        ``sums`` = (sum of band powers (B, F), sum of channel variances
        (B, C), window count (B, 1), short-term EMA of the band powers
        (B, F)) over the earlier windows of each recording, or None at its
        start.
        """
        B, L, C, T = X.shape
        Xf = preprocess(X.reshape(B * L, C, T))
        bp = ((self.band_powers(Xf) - self.bp_mean) / self.bp_std
              ).reshape(B, L, -1)
        if self.use_adapt:
            var = Xf.var(-1).reshape(B, L, C)
            if sums is None:
                sums = (bp.new_zeros(B, bp.shape[-1]), var.new_zeros(B, C),
                        var.new_zeros(B, 1), None)
            count = sums[2].unsqueeze(1) + torch.arange(
                1, L + 1, device=X.device).float().view(1, L, 1)
            cum_bp = sums[0].unsqueeze(1) + bp.cumsum(1)
            cum_var = sums[1].unsqueeze(1) + var.cumsum(1)
            mean = cum_bp / count
            rel = bp - mean
            scale = (cum_var / count).sqrt().clamp_min(
                0.25 * self.chan_std.view(1, 1, C))
            parts_bp = [bp, rel]
            ema = sums[3]
            if self.n_feats >= 2:
                emas = []
                for i in range(L):
                    ema = (bp[:, i] if ema is None
                           else ema + self.ema_alpha * (bp[:, i] - ema))
                    emas.append(ema)
                emas = torch.stack(emas, dim=1)
                parts_bp += [bp - emas, emas - mean]
            sums = (cum_bp[:, -1], cum_var[:, -1], count[:, -1], ema)
            bp_in = torch.cat(parts_bp, dim=-1)
        else:
            scale = self.chan_std.view(1, 1, C).expand(B, L, C)
            bp_in = bp
        parts = []
        if self.use_cnn:
            h = self.cnn(Xf / scale.reshape(B * L, C, 1))
            parts.append(torch.cat([h.mean(-1), h.std(-1)], dim=1
                                   ).reshape(B, L, -1))

        idx = torch.arange(L, device=X.device).float().unsqueeze(0)
        idx = idx + torch.as_tensor(start_index, device=X.device,
                                    dtype=torch.float32).reshape(-1, 1)
        t = torch.log1p(idx.expand(B, L) * self.window_s / 60.0).unsqueeze(-1)
        z = self.in_proj(torch.cat(parts + [self.bp_proj(bp_in), t], dim=-1))
        return z, sums

    def forward(self, X, start_index=0, state=None):
        """Sequences (B, L, C, T) of consecutive windows -> logits (B, L, K).

        ``start_index`` is the index of the first window in its recording
        (int or (B,) tensor) and ``state`` = (GRU state, running sums) is
        carried from the previous chunk of the same recordings.
        """
        h, sums = state if state is not None else (None, None)
        z, sums = self.encode(X, start_index, sums)
        out, h = self.gru(z, h)
        return self.head(out), (h, sums)

    # --- streamed inference ---------------------------------------------

    def step_proba(self, X):
        """Next windows (B, C, T) of the current recording -> p (B, K)."""
        # The B windows of a call are consecutive windows of one recording
        # (B = 1 in streamed evaluation).
        logits, self._state = self(X.float().unsqueeze(0), self._n,
                                   self._state)
        self._n += X.shape[0]
        return logits[0].softmax(-1)

    def reset_state(self):
        self._state = None
        self._n = 0


class SleepOnsetEnsemble(nn.Module):
    """Averages the members' time-to-N2 distributions; predicts a quantile."""

    def __init__(self, n_chans, n_times, sfreq, n_members=N_SEEDS,
                 adapt=ADAPT, use_cnn=USE_CNN, feats=FEATS):
        super().__init__()
        self.members = nn.ModuleList(
            SleepOnsetCNNGRU(n_chans, n_times, sfreq, adapt, use_cnn, feats)
            for _ in range(n_members))
        self.register_buffer("quantile", torch.tensor(0.5))

    def _step_proba(self, X):
        return torch.stack([m.step_proba(X) for m in self.members]).mean(0)

    @torch.inference_mode()
    def predict(self, X):
        self.eval()
        p = self._step_proba(X)
        return quantile_seconds(p, self.quantile.item()).clamp(0.0, CAP_S)

    @torch.inference_mode()
    def predict_dist(self, X):
        """(seconds, p over 5 s bins, hazard) for analysis; advances state."""
        self.eval()
        p = self._step_proba(X)
        seconds = quantile_seconds(p, self.quantile.item()).clamp(0.0, CAP_S)
        return seconds, p, p[..., 0]

    def reset_state(self):
        for m in self.members:
            m.reset_state()


# --- training helpers (local only) ------------------------------------------


def _load_windows(dataset):
    """Read every window once, in dataset order, preprocessed, as float16."""
    from torch.utils.data import DataLoader
    Xs, ys, recs, onsets, subs = [], [], [], [], []
    t0 = time.time()
    for X, y, info in DataLoader(dataset, batch_size=512, shuffle=False):
        Xs.append(preprocess(X.float().cpu()).half())
        ys.append(y.float().cpu().reshape(-1))
        recs.append(np.asarray(info["record_id"]).reshape(-1))
        onsets.append(np.asarray(info["onset"], dtype=np.float64).reshape(-1))
        sid = info.get("subject_id", info["record_id"])
        subs.append(np.asarray(sid).reshape(-1))
    print(f"[CNN-GRU] loaded {sum(len(x) for x in Xs)} windows in "
          f"{time.time() - t0:.0f} s")
    return (torch.cat(Xs), torch.cat(ys), np.concatenate(recs),
            np.concatenate(onsets), np.concatenate(subs))


def _extra_windows(train_dataset):
    """NeuralBench's validation split of the public Muse training release.

    The harness builds NeuralBench's train/val/test loaders but hands the
    solver only the train one; its validation split (20 % of the sleepers
    without a test recording, fixed by ``valid_random_state``) is public
    training data too, so it is loaded here, for training only. Local
    training only: never runs on Codabench. Returns None when unavailable
    (e.g. the Simulated dataset) or if it would overlap the train split.
    """
    if not hasattr(train_dataset, "seg_ds"):
        return None
    try:
        from benchopt.config import get_data_path
        from benchmark_utils.nb_task import load_task
        t0 = time.time()
        loaders, _ = load_task(
            "eeg", "_sleep_onset_stream",
            data_dir=get_data_path("neural_compet"), device="cpu",
            batch_size=512, target_transform=lambda y: y.reshape(()))
    except Exception as err:          # noqa: BLE001 - optional extra data
        print(f"[CNN-GRU] extra data unavailable ({err}); skipping")
        return None
    val_ds = loaders["val"].dataset
    train_tl = {seg.timeline for seg in train_dataset.seg_ds.segments}
    val_tl = {seg.timeline for seg in val_ds.seg_ds.segments}
    if train_tl & val_tl:
        print(f"[CNN-GRU] extra data overlaps the train split "
              f"({len(train_tl & val_tl)} recordings); skipping")
        return None
    print(f"[CNN-GRU] extra split: {len(val_tl)} recordings "
          f"(set up in {time.time() - t0:.0f} s)")
    return _load_windows(val_ds)


def _recordings(y, rec, onset, sub, rng):
    """Window indices of each recording, in time order, plus its subject.

    Data without real recordings (the Simulated set: one record of random
    windows) is cut into pseudo-recordings sorted by decreasing latency.
    """
    out = []
    if len(np.unique(rec)) < 4:
        perm = rng.permutation(len(y))
        for i, part in enumerate(np.array_split(perm, 10)):
            part = part[np.argsort(-y[part].numpy(), kind="stable")]
            out.append((part, i))
        return out
    for r in np.unique(rec):
        idx = np.flatnonzero(rec == r)
        out.append((idx[np.argsort(onset[idx], kind="stable")],
                    int(sub[idx[0]])))
    return out


def _window_weights(y):
    """Per-window loss weights of one recording, mimicking its W-bMAE.

    Each non-empty metric range gets its severity weight split over its
    windows; the recording's weights sum to 1.
    """
    edges = np.asarray(RANGE_EDGES)
    r = np.clip(np.digitize(y, edges[1:-1]), 0, len(RANGE_WEIGHTS) - 1)
    w = np.zeros(len(y))
    present = np.unique(r)
    total = sum(RANGE_WEIGHTS[b] for b in present)
    for b in present:
        mask = r == b
        w[mask] = RANGE_WEIGHTS[b] / total / mask.sum()
    return torch.as_tensor(w, dtype=torch.float32)


def _soft_labels(y, device):
    """Targets (N,) seconds -> Gaussian-smoothed bin distributions (N, K)."""
    centers = torch.arange(N_BINS, device=device).float()
    pos = (y.clamp(0, CAP_S) / BIN_S).unsqueeze(-1)
    logits = -0.5 * ((centers - pos) / LABEL_SIGMA) ** 2
    return logits.softmax(-1)


def _augment(X):
    """Per-recording amplitude scaling, noise and channel dropout.

    X: (B, L, C, T), preprocessed microvolts.
    """
    B, _, C, _ = X.shape
    scale = torch.empty(B, 1, 1, 1, device=X.device).uniform_(0.7, 1.3)
    X = X * scale + torch.randn_like(X) * 2.0
    drop = torch.rand(B, device=X.device) < 0.1
    if drop.any():
        ch = torch.randint(0, C, (B,), device=X.device)
        mask = torch.ones(B, 1, C, 1, device=X.device)
        mask[drop, 0, ch[drop], 0] = 0.0
        X = X * mask
    return X


def _batch_tensor(X_all, recs, start, stop, device):
    """Pad the windows [start, stop) of each recording into (B, L, C, T)."""
    L = min(stop, max(len(idx) for idx in recs)) - start
    X = torch.zeros(len(recs), L, *X_all.shape[1:], dtype=torch.float16)
    mask = torch.zeros(len(recs), L, dtype=torch.bool)
    for b, idx in enumerate(recs):
        part = idx[start:stop]
        X[b, :len(part)] = X_all[part]
        mask[b, :len(part)] = True
    return X.to(device).float(), mask.to(device)


def _detach(state):
    h, sums = state
    return h.detach(), None if sums is None else tuple(
        None if s is None else s.detach() for s in sums)


@torch.no_grad()
def _sequence_proba(member, X_all, idx, device):
    """Distributions (L, K) of one whole recording, chunked, in eval mode."""
    member.eval()
    out, state = [], None
    for s in range(0, len(idx), 4 * CHUNK):
        X, _ = _batch_tensor(X_all, [idx], s, s + 4 * CHUNK, device)
        logits, state = member(X, s, state)
        out.append(logits[0].softmax(-1))
    return torch.cat(out)


def _scores(probas, y_all, recs):
    """W-bMAE averaged over recordings, for every candidate quantile.

    ``probas[i]`` is the (L, K) distribution of recording ``recs[i]``.
    """
    from benchmark_utils.metrics import binned_mae
    scores = {q: [] for q in QUANTILES}
    for p, (idx, _sub) in zip(probas, recs):
        y = y_all[idx].numpy()
        for q in QUANTILES:
            pred = quantile_seconds(p, q).cpu().numpy()
            scores[q].append(binned_mae(y, pred, RANGE_EDGES, RANGE_WEIGHTS))
    return {q: float(np.mean(s)) for q, s in scores.items()}


def _train_device(default):
    forced = os.environ.get("SLEEP_TRAIN_DEVICE")
    if forced:
        return forced
    if default == "cpu" and torch.backends.mps.is_available():
        return "mps"
    return default


def _train_member(model, seed, X_all, y_all, train_recs, val_recs, weights,
                  device, tag):
    """Train one member in place; keep its best epoch on validation."""
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    opt = torch.optim.AdamW(model.parameters(), lr=LR,
                            weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)

    best, best_state, bad = float("inf"), None, 0
    for epoch in range(EPOCHS):
        t0 = time.time()
        model.train()
        order = rng.permutation(len(train_recs))
        # Batch recordings of similar length to limit padding.
        order = sorted(order, key=lambda i: len(train_recs[i][0])
                       + rng.uniform(0, 60))
        batches = [order[i:i + RECS_PER_BATCH]
                   for i in range(0, len(order), RECS_PER_BATCH)]
        rng.shuffle(batches)
        running, n_rec = 0.0, 0
        for batch in batches:
            idxs = [train_recs[i][0] for i in batch]
            max_len = max(len(idx) for idx in idxs)
            y = torch.zeros(len(idxs), max_len)
            w = torch.zeros(len(idxs), max_len)
            for b, idx in enumerate(idxs):
                y[b, :len(idx)] = y_all[idx]
                w[b, :len(idx)] = weights[id(idx)]
            y, w = y.to(device), w.to(device)
            state = None
            for s in range(0, max_len, CHUNK):
                X, mask = _batch_tensor(X_all, idxs, s, s + CHUNK, device)
                logits, state = model(_augment(X), s, state)
                state = _detach(state)
                yc, wc = y[:, s:s + CHUNK], w[:, s:s + CHUNK] * mask
                logp = logits.log_softmax(-1)
                ce = -(_soft_labels(yc, device) * logp).sum(-1)
                mean = (logp.exp() * model.bin_values).sum(-1)
                l1 = (mean - yc.clamp(0, CAP_S)).abs() / CAP_S
                loss = ((ce + L1_WEIGHT * l1) * wc).sum() / len(idxs)
                opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
                running += loss.item() * len(idxs)
            n_rec += len(idxs)
        sched.step()

        probas = [_sequence_proba(model, X_all, idx, device)
                  for idx, _ in val_recs]
        q, score = min(_scores(probas, y_all, val_recs).items(),
                       key=lambda kv: kv[1])
        print(f"[CNN-GRU {tag}] epoch {epoch + 1}/{EPOCHS} "
              f"loss {running / n_rec:.3f} val W-bMAE {score:.1f} s "
              f"(q={q}) {time.time() - t0:.0f} s")
        if score < best:
            best, bad = score, 0
            best_state = {k: v.detach().clone()
                          for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best_state)
    model.eval()
    return best


def _subsample_subjects(recs, frac):
    """Keep the recordings of a random ``frac`` of their subjects."""
    if frac >= 1.0:
        return recs
    subjects = np.unique([s for _, s in recs])
    rng = np.random.default_rng(12345)
    keep = set(rng.choice(subjects, max(1, int(round(frac * len(subjects)))),
                          replace=False))
    return [r for r in recs if r[1] in keep]


def _save_val_predictions(val_recs, rec, sub, y_all, preds):
    """Validation-sleeper predictions -> build/<label>_valpred.npz.

    Training-time analysis only (blend, error per range); written next to
    the exported folder, not inside it, so it never ships in the ZIP.
    """
    sub_dir = os.environ.get("COMPET_SUBMISSION_DIR")
    if not sub_dir:
        return
    sub_dir = os.path.abspath(sub_dir)
    out = os.path.join(os.path.dirname(sub_dir),
                       os.path.basename(sub_dir) + "_valpred.npz")
    idx = np.concatenate([i for i, _ in val_recs])
    # ``record`` = index of the validation recording (identical order in
    # cnn_gru.py and gbm.py), ``record_id`` = the dataset's own id.
    np.savez(out, record=np.concatenate([np.full(len(i), k) for k, (i, _)
                                         in enumerate(val_recs)]),
             record_id=np.asarray(rec[idx]).astype(str),
             subject=np.asarray(sub[idx]).astype(str),
             position=np.concatenate([np.arange(len(i)) for i, _ in val_recs]),
             y=y_all[idx].numpy(), pred=np.concatenate(preds))
    print(f"[CNN-GRU] validation predictions -> {out}")


class Solver(CompetSolver):

    name = "CNN-GRU-Hazard"

    def load_model(self, meta):
        weights = meta["submission_dir"] / WEIGHTS_FILE
        state = None
        # Inference (Codabench): load the shipped weights, which also fix the
        # ensemble size and the adaptation flag. A local training run
        # (train_loader set) starts from scratch instead.
        if weights.exists() and self.train_loader is None:
            state = torch.load(weights, map_location=meta["device"],
                               weights_only=True)
            n_members = len({k.split(".")[1] for k in state
                             if k.startswith("members.")})
            adapt = bool(state["members.0.adapt"])
            # Older checkpoints predate the switch and always had the CNN.
            # Older checkpoints predate these switches: CNN on, feats 1.
            use_cnn = bool(state.get("members.0.cnn_on", True))
            feats = int(state.get("members.0.feats", 1))
            for i in range(n_members):
                state.setdefault(f"members.{i}.cnn_on", torch.tensor(True))
                state.setdefault(f"members.{i}.feats", torch.tensor(1))
        else:
            n_members, adapt, use_cnn, feats = N_SEEDS, ADAPT, USE_CNN, FEATS
        model = SleepOnsetEnsemble(meta["n_chans"], meta["n_times"],
                                   float(meta["sfreq"]), n_members, adapt,
                                   use_cnn, feats)
        if state is not None:
            model.load_state_dict(state)
        return model.to(meta["device"]).eval()

    def fit(self, model, train_loader):
        rng = np.random.default_rng(SEED)
        X_all, y_all, rec, onset, sub = _load_windows(train_loader.dataset)
        recs = _recordings(y_all, rec, onset, sub, rng)

        # Hold out whole subjects for early stopping and the quantile.
        subjects = np.unique([s for _, s in recs])
        n_val = max(1, int(round(VAL_FRACTION * len(subjects))))
        val_subjects = set(rng.choice(subjects, n_val, replace=False))
        train_recs = [r for r in recs if r[1] not in val_subjects]
        val_recs = [r for r in recs if r[1] in val_subjects]
        train_recs = _subsample_subjects(train_recs, TRAIN_FRAC)

        # Extra training-only recordings; the validation sleepers above are
        # drawn before they are added, so validation stays comparable.
        extra = _extra_windows(train_loader.dataset) if EXTRA_DATA else None
        if extra is not None:
            n0, rec_off = len(y_all), int(rec.max()) + 1
            sub_off = int(np.max(sub)) + 1
            X_e, y_e, rec_e, onset_e, sub_e = extra
            extra_recs = [(idx + n0, s + sub_off) for idx, s in
                          _recordings(y_e, rec_e, onset_e, sub_e, rng)]
            X_all = torch.cat([X_all, X_e])
            y_all = torch.cat([y_all, y_e])
            rec = np.concatenate([rec, rec_e + rec_off])
            onset = np.concatenate([onset, onset_e])
            sub = np.concatenate([sub, sub_e + sub_off])
            train_recs = train_recs + extra_recs
            recs = recs + extra_recs
            print(f"[CNN-GRU] + {len(extra_recs)} extra training recordings")

        print(f"[CNN-GRU] {len(train_recs)} train / {len(val_recs)} val "
              f"recordings ({len(subjects)} subjects); "
              f"{len(model.members)} members, "
              f"adapt={model.members[0].use_adapt}, "
              f"cnn={model.members[0].use_cnn}, "
              f"feats={model.members[0].n_feats}, extra={extra is not None}, "
              f"train_frac={TRAIN_FRAC}, seed={MEMBER_SEED}")
        weights = {id(idx): _window_weights(y_all[idx].numpy())
                   for idx, _ in recs}

        # Normalisation stats from the training windows, shared by members.
        train_idx = np.concatenate([idx for idx, _ in train_recs])
        sample = X_all[rng.choice(train_idx, min(20000, len(train_idx)),
                                  replace=False)].float()
        chan_std = sample.std(dim=(0, 2)).clamp_min(1e-3)
        first = model.members[0]
        feats = first.band_powers(sample.to(first.chan_std.device))
        for m in model.members:
            m.chan_std.copy_(chan_std)
            m.bp_mean.copy_(feats.mean(0))
            m.bp_std.copy_(feats.std(0).clamp_min(1e-3))

        out_device = model.quantile.device
        device = _train_device(str(out_device))
        model.to(device)
        print(f"[CNN-GRU] training on {device}")

        member_probas = []
        for i, member in enumerate(model.members):
            best = _train_member(member, 1000 * MEMBER_SEED + i, X_all, y_all,
                                 train_recs,
                                 val_recs, weights, device,
                                 f"{i + 1}/{len(model.members)}")
            print(f"[CNN-GRU] member {i + 1}: best val W-bMAE {best:.1f} s")
            member_probas.append([_sequence_proba(member, X_all, idx, device)
                                  for idx, _ in val_recs])

        ensemble = [torch.stack(ps).mean(0) for ps in zip(*member_probas)]
        q, score = min(_scores(ensemble, y_all, val_recs).items(),
                       key=lambda kv: kv[1])
        model.quantile.fill_(q)
        model.to(out_device).eval()
        print(f"[CNN-GRU] ensemble val W-bMAE {score:.1f} s, q={q:.2f}")
        _save_val_predictions(
            val_recs, rec, sub, y_all,
            [quantile_seconds(p, q).clamp(0, CAP_S).cpu().numpy()
             for p in ensemble])
        self._check_streaming(model, X_all, val_recs[0][0], out_device)

    @staticmethod
    @torch.no_grad()
    def _check_streaming(model, X_all, idx, device):
        """Window-by-window predict must match a whole-sequence forward."""
        idx = idx[:200]
        p = torch.stack([_sequence_proba(m, X_all, idx, device)
                         for m in model.members]).mean(0)
        seq = quantile_seconds(p, model.quantile.item())
        model.reset_state()
        stream = torch.cat([model.predict(X_all[i:i + 1].float().to(device))
                            for i in idx])
        model.reset_state()
        diff = (seq.clamp(0, CAP_S) - stream).abs().max().item()
        print(f"[CNN-GRU] streaming check: max |seq - stream| = {diff:.4f} s")
        if diff > BIN_S:
            raise RuntimeError("streamed predictions differ from the "
                               "sequence forward pass")

    def save_model(self, model, path):
        torch.save(model.state_dict(), path / WEIGHTS_FILE)

In [ ]:
%%writefile /content/EEG_sleep_foundation/submission/gbm.py
"""Track 03 (Sleep Onset) submission: classical ML baseline.

Hand-crafted spectral features per 5 s window + causal context features over
the recording so far -> scikit-learn HistGradientBoostingRegressor trained
with an absolute-error loss and W-bMAE-like sample weights.

Per-window features (per channel unless noted): log absolute and relative
band powers (delta, theta, alpha, sigma, beta, high/EMG), theta/alpha ratio,
spectral entropy, 95 % spectral edge, log variance, and one AF7-AF8 slow
(< 1 Hz) power as an eye-movement proxy.

Context features, computed causally from the windows already seen in the
recording (the streaming state, cleared by ``reset_state``): log elapsed
time, the expanding mean of each feature and the current window's deviation
from it, ~1 min and ~5 min EMAs, and the change since the first 2 minutes.

``SLEEP_GBM_FEATURES=time`` keeps only the elapsed time (a control that
shows how much the EEG adds). ``SLEEP_TRAIN_FRAC`` trains on a fraction of
the training sleepers. The validation sleepers are drawn exactly as in
``cnn_gru.py``, so validation scores are comparable across the two solvers.

Self-contained: numpy, scikit-learn, joblib and torch only.
"""

import os
import time

import joblib
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor

from benchmark_utils.base_solver import CompetSolver
from benchmark_utils.data import to_numpy

CAP_S = 600.0
WEIGHTS_FILE = "weights.joblib"
CLIP_UV = 500.0
BANDS = ((0.5, 4.0), (4.0, 8.0), (8.0, 12.0), (12.0, 15.0), (15.0, 30.0),
         (30.0, 60.0))
RANGE_EDGES = (0.0, 40.0, 90.0, 300.0, 600.0)
RANGE_WEIGHTS = (10.0, 5.0, 3.0, 1.0)

FEATURES = os.environ.get("SLEEP_GBM_FEATURES", "all")      # all | time
TRAIN_FRAC = float(os.environ.get("SLEEP_TRAIN_FRAC", 1.0))
MEMBER_SEED = int(os.environ.get("SLEEP_SEED", 0))
SEED = 0                                 # validation split seed (fixed)
VAL_FRACTION = 0.15                      # same as cnn_gru.py

EMA_FAST_S = 60.0
EMA_SLOW_S = 300.0
BASELINE_S = 120.0

GBM_PARAMS = dict(loss="absolute_error", learning_rate=0.05, max_iter=600,
                  max_leaf_nodes=31, min_samples_leaf=200,
                  l2_regularization=1.0, max_features=0.5)


# --- features ----------------------------------------------------------------


def window_features(X, sfreq):
    """Raw windows (N, C, T) microvolts -> per-window features (N, F)."""
    X = np.asarray(X, dtype=np.float64)
    X = X - X.mean(axis=-1, keepdims=True)
    X = np.clip(X, -CLIP_UV, CLIP_UV)
    N, C, T = X.shape
    freqs = np.fft.rfftfreq(T, d=1.0 / sfreq)
    spec = np.abs(np.fft.rfft(X * np.hanning(T), axis=-1)) ** 2   # (N, C, K)

    nyq = sfreq / 2
    power = np.stack([spec[..., (freqs >= lo) & (freqs < min(hi, nyq))]
                      .sum(-1) for lo, hi in BANDS], axis=-1)       # (N, C, B)
    log_abs = np.log(power + 1e-6)
    log_rel = np.log(power / (power.sum(-1, keepdims=True) + 1e-6) + 1e-6)
    theta_alpha = log_abs[..., 1] - log_abs[..., 2]

    band = (freqs >= 0.5) & (freqs < min(60.0, nyq))
    p = spec[..., band]
    p = p / (p.sum(-1, keepdims=True) + 1e-12)
    entropy = -(p * np.log(p + 1e-12)).sum(-1) / np.log(p.shape[-1])
    edge = freqs[band][np.argmax(p.cumsum(-1) >= 0.95, axis=-1)]
    log_var = np.log(X.var(-1) + 1e-6)

    a, b = (1, 2) if C >= 3 else (0, min(1, C - 1))   # AF7, AF8 on Muse
    eog = X[:, a] - X[:, b]
    eog_spec = np.abs(np.fft.rfft(eog * np.hanning(T), axis=-1)) ** 2
    slow = np.log(eog_spec[:, (freqs > 0) & (freqs < 1.0)].sum(-1) + 1e-6)

    return np.concatenate([
        log_abs.reshape(N, -1), log_rel.reshape(N, -1), theta_alpha, entropy,
        edge, log_var, slow[:, None]], axis=1).astype(np.float32)


class ContextState:
    """Causal per-recording summaries of the window features."""

    def __init__(self, window_s):
        self.window_s = window_s
        self.a_fast = min(1.0, window_s / EMA_FAST_S)
        self.a_slow = min(1.0, window_s / EMA_SLOW_S)
        self.n_base = max(1, int(round(BASELINE_S / window_s)))
        self.reset()

    def reset(self):
        self.n = 0
        self.total = self.fast = self.slow = self.base_sum = None

    def step(self, f):
        """Features of the next window (F,) -> context features (D,)."""
        f = f.astype(np.float64)
        if self.n == 0:
            self.total = np.zeros_like(f)
            self.base_sum = np.zeros_like(f)
            self.fast = f.copy()
            self.slow = f.copy()
        else:
            self.fast += self.a_fast * (f - self.fast)
            self.slow += self.a_slow * (f - self.slow)
        self.n += 1
        self.total += f
        if self.n <= self.n_base:
            self.base_sum += f
        mean = self.total / self.n
        base = self.base_sum / min(self.n, self.n_base)
        elapsed_min = (self.n - 1) * self.window_s / 60.0
        t = np.array([np.log1p(elapsed_min), elapsed_min])
        if FEATURES == "time":
            return t
        return np.concatenate([t, f, f - mean, self.fast, self.slow,
                               self.fast - base, self.slow - base,
                               self.fast - self.slow])

    def run(self, feats):
        """Features (L, F) of consecutive windows -> context (L, D)."""
        return np.stack([self.step(f) for f in feats]).astype(np.float32)


# --- model -------------------------------------------------------------------


class SleepOnsetGBM:
    """Streaming wrapper: window features -> context -> gradient boosting."""

    def __init__(self, sfreq, n_times, estimator=None):
        self.sfreq = float(sfreq)
        self.window_s = n_times / self.sfreq
        self.clf = estimator
        self.state = ContextState(self.window_s)

    def reset_state(self):
        self.state.reset()

    def predict(self, X):
        """Consecutive windows (B, C, T) of one recording -> (B,) seconds."""
        ctx = self.state.run(window_features(to_numpy(X), self.sfreq))
        if self.clf is None:
            return np.full(len(ctx), CAP_S / 2)
        return np.clip(self.clf.predict(ctx), 0.0, CAP_S)


# --- training helpers (local only) -------------------------------------------


def _window_weights(y):
    """Per-window weights of one recording, mimicking its W-bMAE (sum 1)."""
    edges = np.asarray(RANGE_EDGES)
    r = np.clip(np.digitize(y, edges[1:-1]), 0, len(RANGE_WEIGHTS) - 1)
    w = np.zeros(len(y))
    present = np.unique(r)
    total = sum(RANGE_WEIGHTS[b] for b in present)
    for b in present:
        mask = r == b
        w[mask] = RANGE_WEIGHTS[b] / total / mask.sum()
    return w


def _load(dataset, sfreq):
    """Every window once -> features (N, F), y, record, onset, subject."""
    import torch
    from torch.utils.data import DataLoader
    F, ys, recs, onsets, subs = [], [], [], [], []
    t0 = time.time()
    for X, y, info in DataLoader(dataset, batch_size=512, shuffle=False):
        F.append(window_features(X.float().cpu().numpy(), sfreq))
        ys.append(torch.as_tensor(y).float().cpu().reshape(-1).numpy())
        recs.append(np.asarray(info["record_id"]).reshape(-1))
        onsets.append(np.asarray(info["onset"], dtype=np.float64).reshape(-1))
        subs.append(np.asarray(info.get("subject_id", info["record_id"]))
                    .reshape(-1))
    print(f"[GBM] features of {sum(len(f) for f in F)} windows in "
          f"{time.time() - t0:.0f} s")
    return (np.concatenate(F), np.concatenate(ys), np.concatenate(recs),
            np.concatenate(onsets), np.concatenate(subs))


def _recordings(y, rec, onset, sub, rng):
    """Same as cnn_gru._recordings: time-ordered indices + subject."""
    out = []
    if len(np.unique(rec)) < 4:
        perm = rng.permutation(len(y))
        for i, part in enumerate(np.array_split(perm, 10)):
            out.append((part[np.argsort(-y[part], kind="stable")], i))
        return out
    for r in np.unique(rec):
        idx = np.flatnonzero(rec == r)
        out.append((idx[np.argsort(onset[idx], kind="stable")],
                    int(sub[idx[0]])))
    return out


def _subsample_subjects(recs, frac):
    """Same as cnn_gru._subsample_subjects."""
    if frac >= 1.0:
        return recs
    subjects = np.unique([s for _, s in recs])
    rng = np.random.default_rng(12345)
    keep = set(rng.choice(subjects, max(1, int(round(frac * len(subjects)))),
                          replace=False))
    return [r for r in recs if r[1] in keep]


def _stack(recs, feats, y_all, window_s):
    X, y, w = [], [], []
    for idx, _ in recs:
        X.append(ContextState(window_s).run(feats[idx]))
        y.append(np.clip(y_all[idx], 0, CAP_S))
        w.append(_window_weights(y_all[idx]))
    return np.concatenate(X), np.concatenate(y), np.concatenate(w)


def _wbmae(clf, recs, feats, y_all, window_s):
    from benchmark_utils.metrics import binned_mae
    scores, preds = [], []
    for idx, _ in recs:
        p = np.clip(clf.predict(ContextState(window_s).run(feats[idx])),
                    0, CAP_S)
        preds.append(p)
        scores.append(binned_mae(y_all[idx], p, RANGE_EDGES, RANGE_WEIGHTS))
    return float(np.mean(scores)), preds


def _save_val_predictions(val_recs, rec, sub, y_all, preds):
    """Validation predictions -> build/<label>_valpred.npz (not zipped)."""
    sub_dir = os.environ.get("COMPET_SUBMISSION_DIR")
    if not sub_dir:
        return
    sub_dir = os.path.abspath(sub_dir)
    out = os.path.join(os.path.dirname(sub_dir),
                       os.path.basename(sub_dir) + "_valpred.npz")
    idx = np.concatenate([i for i, _ in val_recs])
    # ``record`` = index of the validation recording (identical order in
    # cnn_gru.py and gbm.py), ``record_id`` = the dataset's own id.
    np.savez(out, record=np.concatenate([np.full(len(i), k) for k, (i, _)
                                         in enumerate(val_recs)]),
             record_id=np.asarray(rec[idx]).astype(str),
             subject=np.asarray(sub[idx]).astype(str),
             position=np.concatenate([np.arange(len(i)) for i, _ in val_recs]),
             y=y_all[idx], pred=np.concatenate(preds))
    print(f"[GBM] validation predictions -> {out}")


class Solver(CompetSolver):

    name = "GBM-Context"

    def load_model(self, meta):
        weights = meta["submission_dir"] / WEIGHTS_FILE
        estimator = None
        if weights.exists() and self.train_loader is None:
            estimator = joblib.load(weights)
        return SleepOnsetGBM(meta["sfreq"], meta["n_times"], estimator)

    def fit(self, model, train_loader):
        rng = np.random.default_rng(SEED)
        feats, y_all, rec, onset, sub = _load(train_loader.dataset,
                                              model.sfreq)
        recs = _recordings(y_all, rec, onset, sub, rng)
        subjects = np.unique([s for _, s in recs])
        n_val = max(1, int(round(VAL_FRACTION * len(subjects))))
        val_subjects = set(rng.choice(subjects, n_val, replace=False))
        train_recs = [r for r in recs if r[1] not in val_subjects]
        val_recs = [r for r in recs if r[1] in val_subjects]
        train_recs = _subsample_subjects(train_recs, TRAIN_FRAC)
        print(f"[GBM] {len(train_recs)} train / {len(val_recs)} val "
              f"recordings ({len(subjects)} subjects); features={FEATURES}, "
              f"train_frac={TRAIN_FRAC}, seed={MEMBER_SEED}")

        Xtr, ytr, wtr = _stack(train_recs, feats, y_all, model.window_s)
        print(f"[GBM] training on {Xtr.shape[0]} windows x "
              f"{Xtr.shape[1]} features")

        # Pick the number of boosting rounds on the validation sleepers'
        # W-bMAE, by growing the ensemble with warm_start.
        params = {**GBM_PARAMS, "min_samples_leaf": max(
            5, min(GBM_PARAMS["min_samples_leaf"], len(ytr) // 50))}
        clf = HistGradientBoostingRegressor(
            **params, early_stopping=False, warm_start=True,
            random_state=MEMBER_SEED)
        best, best_iter, t0 = float("inf"), None, time.time()
        for n_iter in range(50, GBM_PARAMS["max_iter"] + 1, 50):
            clf.set_params(max_iter=n_iter)
            clf.fit(Xtr, ytr, sample_weight=wtr)
            score, _ = _wbmae(clf, val_recs, feats, y_all, model.window_s)
            print(f"[GBM] {n_iter} trees: val W-bMAE {score:.1f} s "
                  f"({time.time() - t0:.0f} s)")
            if score < best:
                best, best_iter = score, n_iter

        clf = HistGradientBoostingRegressor(
            **{**params, "max_iter": best_iter}, early_stopping=False,
            random_state=MEMBER_SEED)
        clf.fit(Xtr, ytr, sample_weight=wtr)
        score, preds = _wbmae(clf, val_recs, feats, y_all, model.window_s)
        model.clf = clf
        print(f"[GBM] ensemble val W-bMAE {score:.1f} s, {best_iter} trees")
        _save_val_predictions(val_recs, rec, sub, y_all, preds)

    def save_model(self, model, path):
        joblib.dump(model.clf, path / WEIGHTS_FILE)

In [ ]:
%%writefile /content/EEG_sleep_foundation/submission/submission.py
"""Track 03 (Sleep Onset) submission: a basic MLP regressor.

Codabench only calls ``Solver.load_model(meta)`` and the returned model's
``predict(X)``; ``fit`` and ``save_model`` are local-training hooks used by
``benchopt run ... -o "Sleep-onset[training=True]"`` to produce ``weights.pt``.

Contract (see competition/tracks/sleep_onset/README.md):
- input  X: float tensor (B, C, T) on meta["device"], raw microvolts
- output  : (B,) float latencies in seconds to the first N2 epoch, in [0, 600]

This file must stay self-contained: only ``torch`` and packages from the
competition's requirements.txt may be imported, and no other local modules.
"""

import torch
from torch import nn

from benchmark_utils.base_solver import CompetSolver

CAP_S = 600.0
WEIGHTS_FILE = "weights.pt"

HIDDEN = (256, 64)
DROPOUT = 0.2
EPOCHS = 30
LR = 1e-3
WEIGHT_DECAY = 1e-4


class SleepOnsetMLP(nn.Module):
    """(B, C, T) -> per-channel standardisation -> flatten -> MLP -> (B,).

    The per-channel mean/std are fitted on the training set and stored as
    buffers, so they ship inside ``weights.pt`` with the rest of the model.
    """

    def __init__(self, n_chans, n_times, hidden=HIDDEN, dropout=DROPOUT):
        super().__init__()
        self.register_buffer("chan_mean", torch.zeros(1, n_chans, 1))
        self.register_buffer("chan_std", torch.ones(1, n_chans, 1))
        layers, in_dim = [], n_chans * n_times
        for h in hidden:
            layers += [nn.Linear(in_dim, h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, X):
        X = (X - self.chan_mean) / self.chan_std
        # Fraction of CAP_S in [0, 1].
        return torch.sigmoid(self.net(X.flatten(1))).squeeze(-1)

    @torch.inference_mode()
    def predict(self, X):
        self.eval()
        return self(X.float()) * CAP_S  # (B,) seconds

    def reset_state(self):
        # Stateless: nothing to reset between streamed recordings.
        pass


class Solver(CompetSolver):

    name = "Sleep-MLP"

    def load_model(self, meta):
        model = SleepOnsetMLP(meta["n_chans"], meta["n_times"])
        weights = meta["submission_dir"] / WEIGHTS_FILE
        # Inference (Codabench): load the shipped weights. A local training
        # run (train_loader set) starts from scratch instead.
        if weights.exists() and self.train_loader is None:
            model.load_state_dict(torch.load(
                weights, map_location=meta["device"], weights_only=True))
        return model.to(meta["device"]).eval()

    def fit(self, model, train_loader):
        # Per-channel normalisation stats over the whole training set.
        total, total_sq, count = 0.0, 0.0, 0
        for X, _y, _info in train_loader:
            X = X.float()
            total = total + X.sum(dim=(0, 2))
            total_sq = total_sq + (X ** 2).sum(dim=(0, 2))
            count += X.shape[0] * X.shape[2]
        mean = total / count
        std = (total_sq / count - mean ** 2).clamp_min(1e-12).sqrt()
        model.chan_mean.copy_(mean.view(1, -1, 1))
        model.chan_std.copy_(std.view(1, -1, 1))

        opt = torch.optim.AdamW(
            model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        # L1 matches the MAE-based track metric.
        loss_fn = nn.L1Loss()
        model.train()
        for epoch in range(EPOCHS):
            running, n = 0.0, 0
            for X, y, _info in train_loader:
                opt.zero_grad()
                loss = loss_fn(model(X.float()), y.float().clamp(0, CAP_S) / CAP_S)
                loss.backward()
                opt.step()
                running += loss.item() * len(y)
                n += len(y)
            print(f"[Sleep-MLP] epoch {epoch + 1}/{EPOCHS} "
                  f"train MAE {running / n * CAP_S:.1f} s")
        model.eval()

    def save_model(self, model, path):
        torch.save(model.state_dict(), path / WEIGHTS_FILE)

In [ ]:
%%writefile /content/EEG_sleep_foundation/scripts/build_submission.py
#!/usr/bin/env python3
"""Train submission/submission.py with Benchopt, re-validate it, and ZIP it.

Steps (mirrors competition/tools/debug_submission.py):
1. Training run (``Sleep-onset[training=True]``): Solver.fit trains the model
   and the export writes submission.py + weights.pt into build/<label>/.
2. Inference-only run on the exported folder, exactly as Codabench runs it,
   to catch architecture/checkpoint mismatches.
3. ZIP the folder's files at the archive root into dist/<label>.zip.

Usage:
    python scripts/build_submission.py                 # synthetic, warm-up shapes
    python scripts/build_submission.py --data muse     # public Muse warm-up data
    python scripts/build_submission.py --data muse --solver submission/my_cnn.py
"""

from __future__ import annotations

import argparse
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
TRACK = ROOT / "competition" / "tracks" / "sleep_onset"
DEFAULT_SOLVER = ROOT / "submission" / "submission.py"

DATASETS = {
    # Synthetic data with the real warm-up tensor contract (4 Muse channels,
    # 5 s at 128 Hz). Validates the pipeline only; the score is meaningless.
    "simulated": "Simulated[n_chans=4,n_times=640,n_train=300,n_test=150,sfreq=128]",
    # Public Muse warm-up data; run `benchopt prepare` on it first.
    "muse": "Interaxon2026Muse",
}


def run(cmd: list[str], env: dict[str, str]) -> None:
    print("+", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True, env=env, cwd=ROOT)


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__,
                                     formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--data", choices=DATASETS, default="simulated")
    parser.add_argument("--solver", type=Path, default=DEFAULT_SOLVER,
                        help="solver file to train (default: submission/submission.py)")
    parser.add_argument("--label",
                        help="output name (default: <solver file stem>_<data>)")
    parser.add_argument("--force", action="store_true",
                        help="overwrite an existing build/dist output")
    args = parser.parse_args()

    benchopt = shutil.which("benchopt") or str(Path(sys.executable).with_name("benchopt"))
    if not Path(benchopt).exists():
        raise SystemExit("benchopt not found; activate the sleep_foundation env")

    solver = args.solver.resolve()
    if not solver.is_file():
        raise SystemExit(f"solver file not found: {solver}")
    stem = "mlp" if solver == DEFAULT_SOLVER else solver.stem
    label = args.label or f"{stem}_{args.data}"
    build_dir = ROOT / "build" / label
    zip_path = ROOT / "dist" / f"{label}.zip"
    if build_dir.exists() or zip_path.exists():
        if not args.force:
            raise SystemExit(f"{build_dir} or {zip_path} exists; use --force")
        shutil.rmtree(build_dir, ignore_errors=True)
        zip_path.unlink(missing_ok=True)
    build_dir.mkdir(parents=True)
    zip_path.parent.mkdir(exist_ok=True)

    env = {**os.environ, "COMPET_SUBMISSION_DIR": str(build_dir)}
    common = [benchopt, "run", str(TRACK), "-d", DATASETS[args.data],
              "-r", "1", "--no-cache", "--no-plot", "--no-html",
              "--no-display", "--local"]

    # 1. Train and export.
    run(common + ["-o", "Sleep-onset[training=True]", "-s", str(solver),
                  "--output", str(ROOT / "build" / f"{label}_train.parquet")], env)
    if not (build_dir / "submission.py").is_file():
        raise SystemExit("training run did not export submission.py")
    if not any(p.is_file() and p.name != "submission.py"
               for p in build_dir.iterdir()):
        raise SystemExit("training run did not export any weights file")

    # 2. Reload the exported folder inference-only, as Codabench does.
    run(common + ["-o", "Sleep-onset[training=False]",
                  "-s", str(build_dir / "submission.py"),
                  "--output", str(ROOT / "build" / f"{label}_eval.parquet")], env)

    # 3. ZIP with every file at the archive root.
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for path in sorted(build_dir.iterdir()):
            if path.is_file():
                zf.write(path, path.name)
    print(f"\nReady to upload: {zip_path}")
    with zipfile.ZipFile(zip_path) as zf:
        for info in zf.infolist():
            print(f"  {info.filename:20s} {info.file_size:>12,d} bytes")
    return 0


if __name__ == "__main__":
    sys.exit(main())

## Install dependencies (keeps Colab's CUDA torch)

In [ ]:
!pip install -q -r competition/requirements.txt

neuralbench asks for its folders (CACHE_DIR, SAVE_DIR, DATA_DIR, W&B) on first
use. The harness overrides those paths with its own data folder, so write the
config here and skip the interactive prompt.

In [ ]:
import json
nb_config = Path.home() / ".neuralbench" / "config.json"
if not nb_config.exists():
    base = Path("/content/neuralbench")
    nb_config.parent.mkdir(parents=True, exist_ok=True)
    nb_config.write_text(json.dumps({
        "USER": "colab", "ENTITY_NAME": "colab", "PROJECT_NAME": "neuralbench",
        "SLURM_PARTITION": "", "SLURM_CONSTRAINT": "", "N_CPUS": 2,
        "CLUSTER": None,
        "CACHE_DIR": str(base / "cache"), "SAVE_DIR": str(base / "save"),
        "DATA_DIR": str(base / "data"), "WANDB_HOST": "",
    }, indent=2))
    for sub in ("cache", "save", "data"):
        (base / sub).mkdir(parents=True, exist_ok=True)
print(nb_config.read_text())

## Muse data: from the Drive cache, else download from NEMAR

In [ ]:
DATA = PROJECT / "competition/tracks/sleep_onset/data"
CACHE = DRIVE_DIR / "muse_data.tar" if USE_DRIVE else None

if not (DATA / "neural_compet").exists() and CACHE is not None and CACHE.exists():
    DATA.mkdir(parents=True, exist_ok=True)
    !tar -xf "{CACHE}" -C "{DATA}"
    print("data restored from Drive")

!benchopt prepare competition/tracks/sleep_onset -d Interaxon2026Muse

if CACHE is not None and not CACHE.exists():
    !tar -cf "{CACHE}" -C "{DATA}" neural_compet
    print("data cached in Drive:", CACHE)

## Smoke test on synthetic data (seconds; the score is meaningless)

In [ ]:
!SLEEP_N_SEEDS=1 python scripts/build_submission.py --solver submission/cnn_gru.py --label smoke_cnn --force 2>&1 | grep -E "CNN-GRU\\]|Error|Ready"
!python scripts/build_submission.py --solver submission/gbm.py --label smoke_gbm --force 2>&1 | grep -E "GBM\\] (ensemble|[0-9]+ train)|Error|Ready"

## Experiments

Every run is scored two ways:

* **val W-bMAE**: 15% of the training *sleepers*, held out and identical for
  every run and both solvers. These sleepers are never trained on, so this
  approximates the **unseen** half of the sealed test.
* **test W-bMAE**: the 40 warm-up recordings (sleepers **seen** in training),
  scored inference-only as on Codabench.

**Round 2** (default, `ROUND = 2`) tests the three changes made after round 1,
all on the GRU without the CNN, 3 seeds each:

| Run | Extra recordings | Drowsiness and trend features |
|---|---|---|
| `r2_base` | no | no (= round 1's `cnn_gru_no_cnn`) |
| `r2_extra` | yes | no |
| `r2_feats` | no | yes |
| `r2_feats_extra` | yes | yes (the new default) |

* **Extra recordings**: NeuralBench's own validation sleepers from the public
  training release (about 88 recordings), which the harness's train loader
  leaves out.
* **Drowsiness features**: slow eye-movement power on AF7−AF8, spectral entropy
  and edge, plus each feature's change over the last ~1 min and its trend
  against the recording's mean.

`ROUND = 1` reruns the bottleneck experiments (CNN vs no CNN, learning curve,
GBM). Finished runs are saved in Drive and skipped on a re-run, so after a
disconnect just **Run all** again.

`RUN_PRODUCTION = True` also builds the 5-member submission with the settings
in `PRODUCTION_ENV` (the defaults = no CNN, new features, extra recordings).
To build only the submission, also set `RUN_EXPERIMENTS = False`. Finished
experiments are restored from Drive anyway, but without Drive they would
retrain.

In [ ]:
import os, shutil

ROUND = 2
SEEDS = [0, 1, 2]
RUN_EXPERIMENTS = True      # the experiment runs of ROUND (False: skip them)
RUN_PRODUCTION = False      # build the 5-member submission ensemble
PRODUCTION_LABEL = "gru_ens5"
PRODUCTION_ENV = {"SLEEP_N_SEEDS": "5"}   # defaults: no CNN, feats 2, extra data

GBM, CNN = "submission/gbm.py", "submission/cnn_gru.py"
# Round 1 settings spelled out, since the solver defaults changed after it.
V1 = {"SLEEP_N_SEEDS": "1", "SLEEP_ADAPT": "1", "SLEEP_USE_CNN": "1",
      "SLEEP_FEATS": "1", "SLEEP_EXTRA_DATA": "0"}
GRU = {"SLEEP_N_SEEDS": "1", "SLEEP_ADAPT": "1", "SLEEP_USE_CNN": "0"}
EXPERIMENTS = {
    1: [  # (name,           solver, env)
        ("gbm_time",         GBM, {"SLEEP_GBM_FEATURES": "time"}),
        ("gbm_all",          GBM, {}),
        ("gbm_frac25",       GBM, {"SLEEP_TRAIN_FRAC": "0.25"}),
        ("gbm_frac50",       GBM, {"SLEEP_TRAIN_FRAC": "0.5"}),
        ("cnn_gru_full",     CNN, V1),
        ("cnn_gru_no_cnn",   CNN, {**V1, "SLEEP_USE_CNN": "0"}),
        ("cnn_gru_frac25",   CNN, {**V1, "SLEEP_TRAIN_FRAC": "0.25"}),
        ("cnn_gru_frac50",   CNN, {**V1, "SLEEP_TRAIN_FRAC": "0.5"}),
    ],
    2: [
        ("r2_base",          CNN, {**GRU, "SLEEP_FEATS": "1", "SLEEP_EXTRA_DATA": "0"}),
        ("r2_extra",         CNN, {**GRU, "SLEEP_FEATS": "1", "SLEEP_EXTRA_DATA": "1"}),
        ("r2_feats",         CNN, {**GRU, "SLEEP_FEATS": "2", "SLEEP_EXTRA_DATA": "0"}),
        ("r2_feats_extra",   CNN, {**GRU, "SLEEP_FEATS": "2", "SLEEP_EXTRA_DATA": "1"}),
    ],
}[ROUND]
RUNS = [(f"{name}_s{seed}", solver, {**env, "SLEEP_SEED": str(seed)})
        for name, solver, env in EXPERIMENTS for seed in SEEDS
        if RUN_EXPERIMENTS]
if RUN_PRODUCTION:
    RUNS.append((PRODUCTION_LABEL, CNN, PRODUCTION_ENV))

BUILD = PROJECT / "build"
RESULTS = DRIVE_DIR / "results" if USE_DRIVE else None

def run_files(label):
    return [p for p in BUILD.glob(f"{label}_*") if p.is_file()] + \
           [BUILD / f"{label}.log"]

def restore(label):
    """Copy a finished run back from Drive; True if it was there."""
    if RESULTS is None or not (RESULTS / f"{label}_eval.parquet").exists():
        return False
    for name in ("eval.parquet", "train.parquet", "valpred.npz"):
        src = RESULTS / f"{label}_{name}"
        if src.exists():
            shutil.copy(src, BUILD)
    if (RESULTS / f"{label}.log").exists():
        shutil.copy(RESULTS / f"{label}.log", BUILD)
    return True

def build(label, solver, env_extra):
    env = {**os.environ, **env_extra}
    log_path = BUILD / f"{label}.log"
    cmd = ["python", "scripts/build_submission.py", "--data", "muse",
           "--solver", solver, "--label", label, "--force"]
    with open(log_path, "w") as log:
        proc = subprocess.Popen(cmd, cwd=PROJECT, env=env, text=True,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        for line in proc.stdout:
            log.write(line)
            if any(k in line for k in ("[CNN-GRU", "[GBM]", "Error", "Ready")):
                print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"{label} failed; see {log_path}")
    if RESULTS is not None:
        RESULTS.mkdir(parents=True, exist_ok=True)
        shutil.copytree(BUILD / label, RESULTS / label, dirs_exist_ok=True)
        for f in run_files(label):
            if f.exists():
                shutil.copy(f, RESULTS)
        shutil.copy(PROJECT / "dist" / f"{label}.zip", RESULTS)

for label, solver, env_extra in RUNS:
    if restore(label):
        print(f"===== {label}: restored from Drive")
        continue
    print(f"===== {label}: {env_extra}")
    build(label, solver, env_extra)

## 1. Summary

Mean ± std over seeds. `gap` = test − val: how much easier seen sleepers are
than unseen ones. The constant row is a 35 s constant (the published floor),
scored on the same validation recordings.

In [ ]:
import re
import numpy as np
import pandas as pd

EDGES = np.array([0, 40, 90, 300, 600.0])
WEIGHTS = np.array([10, 5, 3, 1.0])

def range_of(y):
    return np.clip(np.digitize(y, EDGES[1:-1]), 0, len(WEIGHTS) - 1)

def wbmae(y, p):
    """W-bMAE of one recording, as benchmark_utils.metrics.binned_mae."""
    keep = (y >= EDGES[0]) & (y <= EDGES[-1])      # out-of-range targets ignored
    y, p = y[keep], p[keep]
    r, err = range_of(y), np.abs(p - y)
    present = np.unique(r)
    return sum(WEIGHTS[b] * err[r == b].mean() for b in present) / WEIGHTS[present].sum()

def score(df, col="pred"):
    return np.mean([wbmae(g.y.values, g[col].values) for _, g in df.groupby("record")])

def valpred(label):
    f = BUILD / f"{label}_valpred.npz"
    if not f.exists():
        return None
    d = np.load(f)
    return pd.DataFrame({k: d[k] for k in ("record", "subject", "position", "y", "pred")})

def experiment(label):
    return re.sub(r"_s\d+$", "", label)

rows = []
for label, _, _ in RUNS:
    eval_file = BUILD / f"{label}_eval.parquet"
    if not eval_file.exists():
        continue
    d = pd.read_parquet(eval_file).filter(like="objective_").iloc[-1]
    log = (BUILD / f"{label}.log").read_text() if (BUILD / f"{label}.log").exists() else ""
    val = re.findall(r"ensemble val W-bMAE ([\d.]+)", log)
    rows.append({"experiment": experiment(label), "label": label,
                 "val": float(val[-1]) if val else np.nan,
                 "test": d["objective_wbmae_recording_mean"],
                 "test bMAE": d["objective_bmae"], "test MAE": d["objective_mae"]})
runs = pd.DataFrame(rows, columns=["experiment", "label", "val", "test",
                                   "test bMAE", "test MAE"])

summary = runs.groupby("experiment", sort=False).agg(
    seeds=("label", "size"),
    val=("val", "mean"), val_std=("val", "std"),
    test=("test", "mean"), test_std=("test", "std"))
summary["gap (test-val)"] = summary.test - summary.val

any_val = next((valpred(l) for l in runs.label if valpred(l) is not None), None)
if any_val is not None:
    const = any_val.assign(pred=35.0)
    summary.loc["constant_35s"] = {"seeds": 0, "val": score(const)}
summary.round(1)

## 2. Learning curve (round 1 only): val W-bMAE vs share of training sleepers

In [ ]:
import matplotlib.pyplot as plt

FAMILIES = {   # name: (color, {train fraction: experiment})
    "CNN-GRU": ("#2a78d6", {0.25: "cnn_gru_frac25", 0.5: "cnn_gru_frac50", 1.0: "cnn_gru_full"}),
    "GBM":     ("#eb6834", {0.25: "gbm_frac25", 0.5: "gbm_frac50", 1.0: "gbm_all"}),
}
if not any(e in summary.index for _, pts in FAMILIES.values() for e in pts.values()):
    print("No learning-curve runs in this round (ROUND = 1 has them).")
else:
    fig, ax = plt.subplots(figsize=(7, 4.2))
    for name, (color, points) in FAMILIES.items():
        xs = [f for f, e in points.items() if e in summary.index]
        if not xs:
            continue
        m = np.array([summary.loc[points[f], "val"] for f in xs])
        s = np.nan_to_num(np.array([summary.loc[points[f], "val_std"] for f in xs]))
        ax.plot(xs, m, "-o", color=color, lw=2, ms=8, label=name)
        ax.fill_between(xs, m - s, m + s, color=color, alpha=0.15, lw=0)
        ax.annotate(name, (xs[-1], m[-1]), xytext=(8, 0), textcoords="offset points",
                    va="center", color="#333")
    if "constant_35s" in summary.index:
        ax.axhline(summary.loc["constant_35s", "val"], color="#888", lw=1, ls="--")
        ax.annotate("constant 35 s", (0.25, summary.loc["constant_35s", "val"]),
                    xytext=(0, 4), textcoords="offset points", color="#666", fontsize=9)
    ax.set_xscale("log", base=2)
    ax.set_xticks([0.25, 0.5, 1.0], ["25%", "50%", "100%"])
    ax.set_xlim(0.22, 1.35)
    ax.set_xlabel("share of training sleepers used")
    ax.set_ylabel("val W-bMAE, unseen sleepers (s, lower is better)")
    ax.grid(axis="y", color="#e5e5e5", lw=0.8)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.legend(frameon=False, loc="best")
    plt.tight_layout(); plt.show()

## 3. Where does the error come from?

Per range of the true time-to-N2, on the validation recordings (all seeds
pooled): the MAE, and how many seconds of the W-bMAE each range contributes
(the contributions sum to the score). The range with the largest
contribution is where improvements pay off most. The 0–40 s range carries
weight 10, so it usually dominates.

In [ ]:
LABELS = ["0-40 s", "40-90 s", "90-300 s", "300-600 s"]

def breakdown(df):
    mae = {b: [] for b in range(4)}
    contrib = np.zeros(4)
    n_rec = 0
    for _, g in df.groupby(["seed", "record"]):
        g = g[(g.y >= EDGES[0]) & (g.y <= EDGES[-1])]
        r = range_of(g.y.values)
        err = np.abs(g.pred.values - g.y.values)
        present = np.unique(r)
        for b in present:
            mae[b].append(err[r == b].mean())
            contrib[b] += WEIGHTS[b] * err[r == b].mean() / WEIGHTS[present].sum()
        n_rec += 1
    out = {f"MAE {LABELS[b]}": np.mean(mae[b]) if mae[b] else np.nan for b in range(4)}
    out.update({f"share {LABELS[b]}": contrib[b] / n_rec for b in range(4)})
    return out

rows = {}
for exp in summary.index:
    parts = [valpred(l).assign(seed=l) for l in runs.label[runs.experiment == exp]
             if valpred(l) is not None]
    if parts:
        rows[exp] = breakdown(pd.concat(parts))
if any_val is not None:
    rows["constant_35s"] = breakdown(any_val.assign(pred=35.0, seed=0))
pd.DataFrame(rows).T.round(1)

### Timing near onset

Median prediction (s) by true time to N2, on the validation recordings. A
well-timed model follows the diagonal: about 10 at 0–20 s, 50 at 40–60 s and
so on. In round 1 the model squeezed everything below 150 s into 20–60 s, so it
called onset too early.

In [ ]:
BINS = [-1, 20, 40, 60, 90, 150, 300, 601]
cols = {}
for exp in summary.index:
    parts = [valpred(l) for l in runs.label[runs.experiment == exp]
             if valpred(l) is not None]
    if parts:
        d = pd.concat(parts)
        cols[exp] = d.groupby(pd.cut(d.y, BINS), observed=True).pred.median()
timing = pd.DataFrame(cols)
timing.index = [f"{max(0, iv.left):.0f}-{min(600, iv.right):.0f} s" for iv in timing.index]
timing.index.name = "true time to N2"
timing.round(0)

## 4. Blend: GBM + CNN-GRU (round 1 only)

`w·GBM + (1−w)·CNN-GRU`, scored on the validation recordings. These
recordings also pick `w`, so the gain is slightly optimistic; a gain of only a
second or two is noise.

In [ ]:
def blend_curve(a_exp, b_exp):
    curves = []
    for seed in SEEDS:
        a, b = valpred(f"{a_exp}_s{seed}"), valpred(f"{b_exp}_s{seed}")
        if a is None or b is None:
            continue
        m = a.merge(b[["record", "position", "pred"]], on=["record", "position"],
                    suffixes=("_a", "_b"))
        ws = np.linspace(0, 1, 11)
        curves.append([score(m.assign(pred=w * m.pred_a + (1 - w) * m.pred_b))
                       for w in ws])
    if not curves:
        return None
    return pd.Series(np.mean(curves, axis=0), index=np.linspace(0, 1, 11).round(1))

blends = {f"gbm_all + {cnn}": blend_curve("gbm_all", cnn)
          for cnn in ("cnn_gru_full", "cnn_gru_no_cnn")}
blends = pd.DataFrame({k: v for k, v in blends.items() if v is not None})
if not blends.empty:
    blends.index.name = "w (GBM share)"
    for col in blends:
        best = blends[col].idxmin()
        print(f"{col}: best w = {best}, val W-bMAE {blends[col].min():.1f} "
              f"(GBM alone {blends[col][1.0]:.1f}, CNN-GRU alone {blends[col][0.0]:.1f})")
if blends.empty:
    print("No GBM runs in this round (ROUND = 1 has them).")
blends.round(1)

## 5. Decide

**Round 2.** Compare each run with `r2_base`, using validation first (the
unseen sleepers, 60 recordings) and test second. A change counts only if it
beats `r2_base` by more than about the seeds' std.

* **`r2_extra` better:** more data helps, as the round-1 learning curve
  suggested. Keep it on; external data (ANPHY, NSRR) is then worth a try.
* **`r2_feats` better, especially in the 0–40 and 40–90 s shares and in the
  timing table:** the drowsiness features help find onset. Keep them.
* **`r2_feats_extra` best:** build the submission with `RUN_PRODUCTION = True`
  (these are the defaults).
* **One change hurts:** turn it off in `PRODUCTION_ENV`, e.g.
  `"SLEEP_FEATS": "1"` or `"SLEEP_EXTRA_DATA": "0"`.

**Round 1** (`ROUND = 1`): CNN vs no CNN, the learning curve, GBM vs GRU; see
the reading guide at the top of the experiments.

Download all results (tables, logs, validation predictions) as one ZIP:

In [ ]:
import zipfile
from google.colab import files

out = PROJECT / "experiment_results.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as zf:
    for label, _, _ in RUNS:
        for f in run_files(label):
            if f.exists():
                zf.write(f, f.name)
files.download(str(out))
if RUN_PRODUCTION and (PROJECT / "dist" / f"{PRODUCTION_LABEL}.zip").exists():
    files.download(str(PROJECT / "dist" / f"{PRODUCTION_LABEL}.zip"))